#03_gold_kpi_sinistralidade.sql

Gold - KPI sinistralidade (proxy) e utilização

Este notebook garante granularidade de 1 linha por competencia + recortes (uf, segmento, rede_atendimento, cid_grupo, tipo_evento)

##Objetivo
- entrega para BI visão mensal por recortes de negócio

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria tabela kpi_sinistralidade

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.gold.kpi_sinistralidade (
  competencia STRING,
  uf STRING,
  segmento_vinculo STRING,
  rede_atendimento STRING,
  tipo_evento STRING,
  cid_grupo STRING,

  vidas_expostas BIGINT,
  qtd_eventos BIGINT,
  custo_total DECIMAL(18,2),
  custo_pmpm DOUBLE,

  qtd_eventos_custo_neg BIGINT,
  custo_neg_total DECIMAL(18,2),

  gold_build_ts TIMESTAMP
)
USING DELTA;

In [0]:
ALTER TABLE healthcare_dev.gold.kpi_sinistralidade
ADD COLUMNS (
  custo_total_bruto DECIMAL(18,2),
  custo_total_pos DECIMAL(18,2),
  custo_estorno_total DECIMAL(18,2)
);

##Constroe dataset gold

In [0]:
%sql
INSERT OVERWRITE healthcare_dev.gold.kpi_sinistralidade
WITH
vidas AS (
  SELECT
    competencia,
    uf,
    segmento_vinculo,
    COUNT(DISTINCT beneficiario_id) AS vidas_expostas
  FROM healthcare_dev.gold.mart_member_month
  GROUP BY competencia, uf, segmento_vinculo
),
evt AS (
  SELECT
    date_format(e.data_evento, 'yyyy-MM') AS competencia,
    b.uf,
    b.segmento_vinculo,
    e.rede_atendimento,
    e.tipo_evento,
    e.cid_grupo,
    COUNT(*) AS qtd_eventos,

    -- custo bruto
    SUM(COALESCE(e.valor_pago,0)) AS custo_total_bruto,

    -- custo "BI"
    SUM(CASE WHEN COALESCE(e.valor_pago,0) > 0 THEN COALESCE(e.valor_pago,0) ELSE 0 END) AS custo_total_pos,

    -- estornos/ajustes 
    SUM(CASE WHEN COALESCE(e.valor_pago,0) < 0 THEN COALESCE(e.valor_pago,0) ELSE 0 END) AS custo_estorno_total,
    SUM(CASE WHEN COALESCE(e.valor_pago,0) < 0 THEN 1 ELSE 0 END) AS qtd_eventos_custo_neg,
    SUM(CASE WHEN COALESCE(e.valor_pago,0) < 0 THEN COALESCE(e.valor_pago,0) ELSE 0 END) AS custo_neg_total

  FROM healthcare_dev.silver.fact_eventos e
  JOIN healthcare_dev.silver.dim_beneficiarios b
    ON e.beneficiario_id = b.beneficiario_id
  GROUP BY
    date_format(e.data_evento, 'yyyy-MM'),
    b.uf, b.segmento_vinculo,
    e.rede_atendimento, e.tipo_evento, e.cid_grupo
)
SELECT
  e.competencia,
  e.uf,
  e.segmento_vinculo,
  e.rede_atendimento,
  e.tipo_evento,
  e.cid_grupo,

  v.vidas_expostas,
  e.qtd_eventos,

  e.custo_total_pos AS custo_total,

  CASE WHEN v.vidas_expostas = 0 THEN 0.0
       ELSE ROUND(CAST(e.custo_total_pos AS DOUBLE) / v.vidas_expostas, 4)
  END AS custo_pmpm,

  e.qtd_eventos_custo_neg,
  e.custo_neg_total,

  -- colunas auditoria
  e.custo_total_bruto,
  e.custo_total_pos,
  e.custo_estorno_total,

  current_timestamp() AS gold_build_ts
FROM evt e
LEFT JOIN vidas v
  ON e.competencia = v.competencia
 AND e.uf = v.uf
 AND e.segmento_vinculo = v.segmento_vinculo;